# 08 · Silver — `ai_classify` + `ai_extract`

**Silver job:** turn parse text into typed columns and land a table you can join.

- `ai_classify` labels the packet page: FNOL / invoice / adjuster / policy
- `ai_extract` pulls claim, pet, clinic, visit date, invoice, amount, diagnosis

Labels for `ai_classify` are a **JSON array string**, not `ARRAY()`. Both functions use version **2.0**. `extracted:response:<field>` is the cell to read.

Cap input with `substr(..., 1, 8000)` so a long parse cannot blow the context window.


In [ ]:
%run ./00_setup


In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE silver_claim_documents
COMMENT 'Classified + extracted pet-claim PDF fields'
AS
SELECT
  claim_id,
  file_name,
  path,
  parsed_text,
  ai_classify(
    substr(parsed_text, 1, 4000),
    '["fnol_claim_form", "clinic_invoice", "adjuster_report", "policy_declarations", "other"]'
  ) AS classified,
  ai_extract(
    substr(parsed_text, 1, 8000),
    '{
      "type": "object",
      "properties": {
        "claim_number": {"type": "string", "description": "Claim number such as CLM-PET-00021"},
        "policy_number": {"type": "string", "description": "Policy number such as POL-88421"},
        "pet_name": {"type": "string", "description": "Name of the pet"},
        "species": {"type": "string", "description": "Dog or Cat"},
        "breed": {"type": "string", "description": "Breed of the pet"},
        "member_name": {"type": "string", "description": "Policyholder or clinic client name"},
        "clinic_name": {"type": "string", "description": "Veterinary clinic or hospital name"},
        "visit_date": {"type": "string", "description": "Visit date or date of loss as YYYY-MM-DD"},
        "invoice_number": {"type": "string", "description": "Clinic invoice number if this is an invoice"},
        "billed_amount": {"type": "number", "description": "Total billed amount as a number, no currency symbol"},
        "diagnosis": {"type": "string", "description": "Diagnosis or presenting complaint"},
        "recommended_decision": {"type": "string", "description": "Pay, Deny, or unknown"}
      }
    }'
  ) AS extracted
FROM bronze_parsed_text
""")


In [ ]:
def fld(col, name):
    return f"""coalesce(
      {col}:response:{name}:value::string,
      {col}:response:{name}::string
    )"""

spark.sql(f"""
CREATE OR REPLACE TABLE silver_claim_documents_typed
COMMENT 'Silver typed columns from ai_classify + ai_extract'
AS
SELECT
  claim_id AS packet_claim_id,
  file_name,
  path,
  coalesce(
    classified:response:result::string,
    classified:result::string,
    trim(both '"' from CAST(classified AS STRING))
  ) AS doc_type,
  {fld('extracted','claim_number')} AS claim_number,
  {fld('extracted','policy_number')} AS policy_number,
  {fld('extracted','pet_name')} AS pet_name,
  {fld('extracted','species')} AS species,
  {fld('extracted','breed')} AS breed,
  {fld('extracted','member_name')} AS member_name,
  {fld('extracted','clinic_name')} AS clinic_name,
  try_to_date({fld('extracted','visit_date')}) AS visit_date,
  {fld('extracted','invoice_number')} AS invoice_number,
  try_cast({fld('extracted','billed_amount')} AS DOUBLE) AS billed_amount,
  {fld('extracted','diagnosis')} AS diagnosis,
  {fld('extracted','recommended_decision')} AS recommended_decision
FROM silver_claim_documents
""")

display(spark.sql("""
SELECT
  packet_claim_id, doc_type, claim_number, pet_name, clinic_name,
  visit_date, invoice_number, billed_amount, diagnosis, recommended_decision
FROM silver_claim_documents_typed
ORDER BY packet_claim_id, doc_type
"""))


### Spot-check

You should see four `clinic_invoice` rows. Max appears twice at Pine Vet Clinic on `2026-08-03` with **different** invoice numbers. That is the planted duplicate.


In [ ]:
display(spark.sql("""
SELECT pet_name, clinic_name, visit_date, count(*) AS invoice_rows,
       collect_list(invoice_number) AS invoices,
       collect_list(claim_number) AS claims
FROM silver_claim_documents_typed
WHERE doc_type = 'clinic_invoice'
GROUP BY pet_name, clinic_name, visit_date
ORDER BY invoice_rows DESC, pet_name
"""))
